# 🎬 AI Dubbing Service (Colab Edition)
**Upload a video → Get it dubbed in any language with Original Voice Cloning!**

This notebook runs the full dubbing pipeline on Google Colab's free T4 GPU (16GB VRAM).

---

## Step 1: Check GPU (Should show Tesla T4)

In [ ]:
!nvidia-smi

## Step 2: Install Dependencies (~3-5 minutes)

In [ ]:
# Install system dependencies
!sudo apt-get -y update
!sudo apt-get -y install espeak-ng build-essential cmake libsndfile1

# Upgrade build tools
!pip install -U pip setuptools wheel

# Install TTS from source (bypassing build isolation to fix Py3.12 issues)
!pip install --no-build-isolation git+https://github.com/coqui-ai/TTS

# Install other dependencies
!pip install -q faster-whisper deep-translator google-generativeai ffmpeg-python edge-tts gtts demucs flask pyngrok

print("\n✅ All packages installed!")

## Step 3: Upload Your Video
Run the cell below and **select your video file** from your computer.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
INPUT_VIDEO = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {INPUT_VIDEO} ({os.path.getsize(INPUT_VIDEO) / 1024 / 1024:.1f} MB)")

## Step 4: Configure Settings
Change the language and API key below.

In [ ]:
# ========== SETTINGS ==========
TARGET_LANGUAGE = "hi"  # Change this: hi=Hindi, en=English, es=Spanish, fr=French, de=German, ja=Japanese, ko=Korean
GEMINI_API_KEY = ""     # Optional: Paste your Gemini API key for text polishing (leave empty to skip)
# ==============================

OUTPUT_VIDEO = f"dubbed_{TARGET_LANGUAGE}_{INPUT_VIDEO}"
print(f"🎯 Target Language: {TARGET_LANGUAGE}")
print(f"📁 Output: {OUTPUT_VIDEO}")

## Step 5: Run the Dubbing Engine 🚀
This is the main cell. It will:
1. Extract audio
2. Separate vocals (Demucs)
3. Transcribe speech (Whisper)
4. Translate text
5. Clone voice (XTTS) ← This is what your GTX 1650 couldn't handle!
6. Mix and merge everything

In [ ]:
import torch
import ffmpeg
import subprocess
import shutil
import os
import gc
import asyncio
import edge_tts
from deep_translator import GoogleTranslator
from faster_whisper import WhisperModel
from gtts import gTTS
import sys

# Voice mapping for Edge TTS fallback
VOICE_MAPPING = {
    "hi": "hi-IN-SwaraNeural",
    "en": "en-US-JennyNeural",
    "es": "es-ES-ElviraNeural",
    "fr": "fr-FR-DeniseNeural",
    "de": "de-DE-KatjaNeural",
    "it": "it-IT-ElsaNeural",
    "pt": "pt-BR-FranciscaNeural",
    "zh-cn": "zh-CN-XiaoxiaoNeural",
    "ja": "ja-JP-NanamiNeural",
    "ko": "ko-KR-SunHiNeural",
    "ru": "ru-RU-SvetlanaNeural"
}

async def generate_edge_tts(text, voice, output_file):
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(output_file)

def get_audio_duration(file_path):
    try:
        probe = ffmpeg.probe(file_path)
        return float(probe['format']['duration'])
    except:
        return 0

def polish_text_with_gemini(text, target_language, api_key):
    if not text or len(text.strip()) < 2 or not api_key:
        return text
    try:
        import google.generativeai as genai
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel('gemini-pro')
        prompt = f'Refine this text for a video dubbing script in {target_language}. Rules: 1. Correct grammar. 2. Make it natural. 3. Do NOT change meaning. 4. Output ONLY the polished text. Input: \"{text}\"'
        response = model.generate_content(prompt)
        if response.text:
            cleaned = response.text.strip().replace('"', '').replace('\\n', ' ')
            if len(cleaned) < len(text) * 3:
                print(f"  ✨ Polished: '{text}' -> '{cleaned}'")
                return cleaned
        return text
    except Exception as e:
        print(f"  ⚠️ Gemini: {e}")
        return text

# ===== MAIN PIPELINE =====
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Device: {device} | GPU: {torch.cuda.get_device_name(0) if device == 'cuda' else 'N/A'}")
print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if device == 'cuda' else '')

work_dir = "dubbing_work"
if os.path.exists(work_dir): shutil.rmtree(work_dir)
os.makedirs(work_dir)
chunks_dir = os.path.join(work_dir, "chunks")
os.makedirs(chunks_dir)

# Step 1: Extract Audio
print("\n🎵 Step 1: Extracting Audio...")
full_audio = os.path.join(work_dir, "full_source.wav")
ffmpeg.input(INPUT_VIDEO).output(full_audio, ac=1, ar=24000).run(cmd="ffmpeg", overwrite_output=True, quiet=True)
total_duration = get_audio_duration(full_audio)
print(f"  Duration: {total_duration:.1f}s")

# Step 2: Vocal Separation (Demucs)
print("\n🎤 Step 2: Separating Vocals (Demucs)...")
demucs_out = os.path.join(work_dir, "demucs_out")
subprocess.run(["demucs", "--two-stems=vocals", "-n", "htdemucs", "-d", device, "--shifts", "0", "--overlap", "0.1", full_audio, "-o", demucs_out], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
vocals_path = os.path.join(demucs_out, "htdemucs", "full_source", "vocals.wav")
background_path = os.path.join(demucs_out, "htdemucs", "full_source", "no_vocals.wav")
print("  ✅ Vocals separated!")
gc.collect(); torch.cuda.empty_cache()

# Step 3: Transcription (Whisper)
print("\n📝 Step 3: Transcribing (Whisper small)...")
whisper_model = WhisperModel("small", device="cpu", compute_type="int8")
segments_gen, info = whisper_model.transcribe(vocals_path, beam_size=1, vad_filter=True, vad_parameters=dict(min_silence_duration_ms=500))
segments = [{"start": s.start, "end": s.end, "text": s.text} for s in segments_gen]
print(f"  Found {len(segments)} segments.")
for i, s in enumerate(segments[:5]):
    print(f"    [{i}] {s['start']:.1f}-{s['end']:.1f}s: {s['text']}")
del whisper_model; gc.collect()

# Step 4: Load XTTS (Voice Cloning)
print("\n🧬 Step 4: Loading XTTS v2 (Voice Cloning)...")
xtts_supported = ["en", "es", "fr", "de", "it", "pt", "pl", "tr", "ru", "nl", "cs", "ar", "zh-cn", "ja", "hu", "ko", "hi"]
use_xtts = TARGET_LANGUAGE in xtts_supported

tts_engine = None
if use_xtts:
    from TTS.api import TTS
    from TTS.tts.configs.xtts_config import XttsConfig
    from TTS.config.shared_configs import BaseDatasetConfig, BaseAudioConfig
    from TTS.tts.models.xtts import XttsAudioConfig, XttsArgs
    old_stdout = sys.stdout; sys.stdout = open(os.devnull, 'w')
    with torch.serialization.safe_globals([XttsConfig, BaseDatasetConfig, BaseAudioConfig, XttsAudioConfig, XttsArgs]):
        tts_engine = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
    sys.stdout = old_stdout
    print("  ✅ XTTS loaded on GPU!")
else:
    print(f"  ℹ️ Language '{TARGET_LANGUAGE}' not supported by XTTS. Using Edge TTS.")

# Master Reference
longest_seg = max(segments, key=lambda s: s['end'] - s['start']) if segments else None
master_ref = os.path.join(chunks_dir, "master_ref.wav")
if longest_seg and (longest_seg['end'] - longest_seg['start']) > 3:
    subprocess.run(f'ffmpeg -i "{vocals_path}" -ss {longest_seg["start"]} -t {longest_seg["end"]-longest_seg["start"]} -ac 1 -ar 24000 "{master_ref}" -y -loglevel error', shell=True)
else:
    master_ref = None

# Step 5: Dub Each Segment
print(f"\n🎙️ Step 5: Dubbing {len(segments)} segments...")
temp_files = [None] * len(segments)
current_time = 0.0

for i, seg in enumerate(segments):
    start, end, text = seg['start'], seg['end'], seg['text']
    duration = end - start

    # Reference audio
    ref_path = os.path.join(chunks_dir, f"ref_{i}.wav")
    use_master = False
    if duration < 2.5 and master_ref and os.path.exists(master_ref):
        ref_path = master_ref; use_master = True
    else:
        subprocess.run(f'ffmpeg -i "{vocals_path}" -ss {start} -t {duration} -ac 1 -ar 24000 "{ref_path}" -y -loglevel error', shell=True)

    # Gap
    gap_dur = start - current_time
    gap_path = None
    if gap_dur > 0.1:
        gap_path = os.path.join(chunks_dir, f"gap_{i}.wav")
        subprocess.run(f'ffmpeg -f lavfi -i anullsrc=r=24000:cl=mono -t {gap_dur} {gap_path} -y -loglevel error', shell=True)

    # Translate
    try:
        translated = GoogleTranslator(source='auto', target=TARGET_LANGUAGE).translate(text)
        if GEMINI_API_KEY:
            translated = polish_text_with_gemini(translated, TARGET_LANGUAGE, GEMINI_API_KEY)
    except:
        translated = text

    tts_path = os.path.join(chunks_dir, f"tts_{i}.wav")

    # TTS with XTTS (Clone) -> Edge TTS fallback -> gTTS fallback
    if use_xtts and tts_engine:
        try:
            tts_engine.tts_to_file(text=translated, speaker_wav=ref_path, language=TARGET_LANGUAGE, file_path=tts_path)
            print(f"  [{i+1}/{len(segments)}] ✅ XTTS Cloned: {translated[:40]}...")
        except Exception as e:
            print(f"  [{i+1}/{len(segments)}] ⚠️ XTTS failed, using Edge TTS: {e}")
            voice = VOICE_MAPPING.get(TARGET_LANGUAGE, 'en-US-JennyNeural')
            try:
                await generate_edge_tts(translated, voice, tts_path)
            except:
                gTTS(text=translated, lang=TARGET_LANGUAGE, slow=False).save(tts_path)
    else:
        voice = VOICE_MAPPING.get(TARGET_LANGUAGE, 'en-US-JennyNeural')
        try:
            await generate_edge_tts(translated, voice, tts_path)
            print(f"  [{i+1}/{len(segments)}] ✅ Edge TTS: {translated[:40]}...")
        except:
            gTTS(text=translated, lang=TARGET_LANGUAGE, slow=False).save(tts_path)

    # Speed sync
    actual_dur = get_audio_duration(tts_path)
    if actual_dur == 0: actual_dur = 0.1
    speed = max(0.75, min(actual_dur / duration, 1.3))
    synced_path = os.path.join(chunks_dir, f"synced_{i}.wav")
    subprocess.run(f'ffmpeg -i "{tts_path}" -filter:a "atempo={speed}" "{synced_path}" -y -loglevel error', shell=True)

    temp_files[i] = {"gap": gap_path, "tts": synced_path}
    current_time = end

# Trailing silence
if current_time < total_duration:
    gap_end = os.path.join(chunks_dir, "gap_end.wav")
    subprocess.run(f'ffmpeg -f lavfi -i anullsrc=r=24000:cl=mono -t {total_duration - current_time} {gap_end} -y -loglevel error', shell=True)
    temp_files.append({"gap": None, "tts": gap_end})

# Step 6: Merge
print("\n🔧 Step 6: Merging audio...")
ordered = []
for item in temp_files:
    if item and item['gap']: ordered.append(item['gap'])
    if item and item['tts']: ordered.append(item['tts'])

concat_list = os.path.join(work_dir, "concat.txt")
with open(concat_list, "w") as f:
    for p in ordered:
        f.write(f"file '{p}'\n")

full_tts = os.path.join(work_dir, "full_tts.wav")
subprocess.run(f'ffmpeg -f concat -safe 0 -i "{concat_list}" -c copy "{full_tts}" -y -loglevel error', shell=True)

final_audio = os.path.join(work_dir, "final_mixed.wav")
subprocess.run(f'ffmpeg -i {full_tts} -i {background_path} -filter_complex amix=inputs=2:duration=first {final_audio} -y -loglevel error', shell=True)

# Final video
subprocess.run(f'ffmpeg -i "{INPUT_VIDEO}" -i "{final_audio}" -map 0:v -map 1:a -c:v copy -c:a aac -strict experimental "{OUTPUT_VIDEO}" -y -loglevel error', shell=True)
print(f"\n🎉 DONE! Output: {OUTPUT_VIDEO}")

## Step 6: Download Your Dubbed Video

In [ ]:
from google.colab import files
files.download(OUTPUT_VIDEO)
print("✅ Download started!")